# 18. MC进阶：亚式期权定价

## 学习目标

通过本次学习，你将能够：

1. **理解亚式期权的特点与分类**
2. **区分算术平均与几何平均亚式期权**
3. **用 MC 方法定价路径依赖期权**
4. **对比不同观察频率对期权价格的影响**
5. **理解 MC 方法对复杂衍生品的优势**

## 知识地图

```
亚式期权
├── 定义与特点
│   ├── 路径依赖期权
│   ├── 平均价格期权 (Average Price)
│   └── 平均行权价期权 (Average Strike)
├── 平均方式
│   ├── 算术平均：无解析解
│   └── 几何平均：有解析解
├── 观察频率
│   ├── 连续观察（理论）
│   ├── 日均（每日收盘价）
│   ├── 周均（每周收盘价）
│   └── 月均（每月收盘价）
├── 定价方法
│   ├── 蒙特卡洛模拟（主要方法）
│   ├── 几何平均解析解
│   └── Turnbull-Wakeman 近似
└── 应用场景
    ├── 对冲汇率风险
    ├── 商品价格风险管理
    └── 员工股票期权
```

## 环境依赖

```bash
pip install numpy scipy matplotlib
```

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子
np.random.seed(42)

%matplotlib inline

---
## 1. 理论基础：亚式期权

### 1.1 什么是亚式期权？

亚式期权（Asian Option）是一种**路径依赖期权**，其 payoff 取决于标的资产在一段时间内的**平均价格**，而不是到期时的价格。

### 1.2 亚式期权的分类

| 类型 | Payoff | 说明 |
|------|--------|------|
| **平均价格 Call** | $\max(\bar{S} - K, 0)$ | 平均价格 vs 固定行权价 |
| **平均价格 Put** | $\max(K - \bar{S}, 0)$ | 固定行权价 vs 平均价格 |
| **平均行权价 Call** | $\max(S_T - \bar{S}, 0)$ | 到期价格 vs 平均价格 |
| **平均行权价 Put** | $\max(\bar{S} - S_T, 0)$ | 平均价格 vs 到期价格 |

其中 $\bar{S}$ 是标的资产在观察期内的平均价格。

### 1.3 算术平均 vs 几何平均

| 平均方式 | 公式 | 解析解 | 特点 |
|----------|------|--------|------|
| **算术平均** | $\bar{S} = \frac{1}{n} \sum_{i=1}^{n} S_{t_i}$ | 无 | 更符合实际，但无解析解 |
| **几何平均** | $\bar{S} = \left(\prod_{i=1}^{n} S_{t_i}\right)^{1/n}$ | 有 | 数学上更易处理 |

**为什么几何平均有解析解？**
- 几何平均的对数是正态随机变量的线性组合
- 因此几何平均本身是对数正态分布
- 可以用类似 BS 公式的方法推导解析解

### 1.4 亚式期权的优势

| 优势 | 说明 |
|------|------|
| **降低波动率** | 平均化减少了价格波动的影响 |
| **更便宜** | 比同等欧式期权便宜（波动率降低） |
| **防止操纵** | 难以通过操纵到期日价格来获利 |
| **风险管理** | 适合对冲一段时间内的平均风险 |

---
## 2. 几何平均亚式期权的解析解

### 2.1 解析公式

几何平均亚式期权有解析解（Kemna & Vorst, 1990）：

$$C_{geo} = e^{-rT} \left[ F_0 \cdot N(d_1) - K \cdot N(d_2) \right]$$

其中：
$$F_0 = S_0 \cdot \exp\left(\frac{1}{2}\left(r - \frac{\sigma^2}{6}\right)T\right)$$

$$d_1 = \frac{\ln(F_0/K) + \frac{\sigma_G^2}{2}T}{\sigma_G \sqrt{T}}$$

$$d_2 = d_1 - \sigma_G \sqrt{T}$$

$$\sigma_G = \frac{\sigma}{\sqrt{3}}$$

### 2.2 直觉理解

- 几何平均的波动率是原波动率的 $1/\sqrt{3}$
- 这意味着几何平均亚式期权比欧式期权便宜约 30-40%
- 几何平均总是 ≤ 算术平均（AM-GM 不等式）

In [ ]:
def asian_geometric_call(S0, K, r, sigma, T):
    """
    几何平均亚式看涨期权的解析解
    
    Parameters
    ----------
    S0 : float - 初始价格
    K : float - 行权价
    r : float - 无风险利率
    sigma : float - 波动率
    T : float - 到期时间
    
    Returns
    -------
    float : 期权价格
    """
    # 几何平均的等价波动率
    sigma_G = sigma / np.sqrt(3)
    
    # 远期价格
    F0 = S0 * np.exp(0.5 * (r - sigma**2 / 6) * T)
    
    # d1 和 d2
    d1 = (np.log(F0 / K) + 0.5 * sigma_G**2 * T) / (sigma_G * np.sqrt(T))
    d2 = d1 - sigma_G * np.sqrt(T)
    
    # 期权价格
    price = np.exp(-r * T) * (F0 * norm.cdf(d1) - K * norm.cdf(d2))
    
    return price


# 对比欧式 Call 和几何平均亚式 Call
def bs_call(S, K, r, sigma, T):
    """BS 看涨期权公式"""
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


# 参数设置
S0 = 100
K = 100
r = 0.05
sigma = 0.2
T = 1.0

european_call = bs_call(S0, K, r, sigma, T)
geometric_asian = asian_geometric_call(S0, K, r, sigma, T)

print(f"期权价格对比：")
print(f"  参数: S0={S0}, K={K}, r={r}, σ={sigma}, T={T}")
print(f"\n  欧式 Call:            {european_call:.4f}")
print(f"  几何平均亚式 Call:    {geometric_asian:.4f}")
print(f"  价格差异:            {european_call - geometric_asian:.4f}")
print(f"  相对差异:            {(european_call - geometric_asian) / european_call * 100:.1f}%")
print(f"\n解读：")
print(f"  - 几何平均亚式期权比欧式期权便宜")
print(f"  - 原因：平均化降低了波动率（σ_G = σ/√3 = {sigma/np.sqrt(3):.4f}）")

---
## 3. 蒙特卡洛定价亚式期权

### 3.1 为什么用 MC？

**算术平均亚式期权没有解析解**，这是 MC 方法大显身手的场景！

MC 定价步骤：
1. 模拟 N 条资产价格路径
2. 计算每条路径的平均价格
3. 计算 payoff
4. 折现取平均

### 3.2 实现 MC 定价

In [ ]:
def mc_asian_option(S0, K, r, sigma, T, N, M=252, option_type='call', avg_type='arithmetic'):
    """
    蒙特卡洛定价亚式期权
    
    Parameters
    ----------
    S0 : float - 初始价格
    K : float - 行权价
    r : float - 无风险利率
    sigma : float - 波动率
    T : float - 到期时间
    N : int - 模拟路径数量
    M : int - 观察次数（时间步数）
    option_type : str - 'call' 或 'put'
    avg_type : str - 'arithmetic' 或 'geometric'
    
    Returns
    -------
    dict : 包含价格、标准误差、置信区间
    """
    dt = T / M
    
    # 生成随机数
    Z = np.random.standard_normal((N, M))
    
    # 模拟路径
    paths = np.zeros((N, M + 1))
    paths[:, 0] = S0
    
    for t in range(M):
        paths[:, t + 1] = paths[:, t] * np.exp(
            (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t]
        )
    
    # 计算平均价格（不包括初始价格）
    price_paths = paths[:, 1:]  # 排除初始价格
    
    if avg_type == 'arithmetic':
        avg_prices = price_paths.mean(axis=1)
    elif avg_type == 'geometric':
        # 几何平均：exp(mean(log(S)))
        avg_prices = np.exp(np.log(price_paths).mean(axis=1))
    else:
        raise ValueError("avg_type must be 'arithmetic' or 'geometric'")
    
    # 计算 payoff
    if option_type == 'call':
        payoffs = np.maximum(avg_prices - K, 0)
    elif option_type == 'put':
        payoffs = np.maximum(K - avg_prices, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    
    # 计算价格
    discount = np.exp(-r * T)
    price = discount * payoffs.mean()
    std_error = discount * payoffs.std() / np.sqrt(N)
    
    return {
        'price': price,
        'std_error': std_error,
        'ci_lower': price - 1.96 * std_error,
        'ci_upper': price + 1.96 * std_error,
        'avg_prices': avg_prices
    }


# 定价算术平均亚式 Call
N_mc = 100000
np.random.seed(42)
arithmetic_asian = mc_asian_option(S0, K, r, sigma, T, N_mc, avg_type='arithmetic')

# 定价几何平均亚式 Call（MC）
np.random.seed(42)
geometric_asian_mc = mc_asian_option(S0, K, r, sigma, T, N_mc, avg_type='geometric')

print(f"亚式期权定价对比（N = {N_mc:,}）：\n")
print(f"{'类型':<20} {'价格':>10} {'标准误差':>10} {'与解析解差异':>12}")
print("-" * 55)
print(f"{'欧式 Call':<20} {european_call:>10.4f} {'--':>10} {'--':>12}")
print(f"{'几何平均亚式（解析）':<20} {geometric_asian:>10.4f} {'--':>10} {'--':>12}")
print(f"{'几何平均亚式（MC）':<20} {geometric_asian_mc['price']:>10.4f} {geometric_asian_mc['std_error']:>10.4f} ", end="")
print(f"{abs(geometric_asian_mc['price'] - geometric_asian):>11.4f}")
print(f"{'算术平均亚式（MC）':<20} {arithmetic_asian['price']:>10.4f} {arithmetic_asian['std_error']:>10.4f} {'无解析解':>12}")

print(f"\n解读：")
print(f"  - 算术平均 > 几何平均（AM-GM 不等式）")
print(f"  - MC 几何平均与解析解高度一致，验证了实现正确性")
print(f"  - 亚式期权比欧式期权便宜（平均化降低波动率）")

---
## 4. 对偶变量法定价亚式期权

### 4.1 方差缩减

亚式期权的 MC 定价同样可以使用方差缩减技术。我们实现对偶变量法版本：

In [ ]:
def mc_asian_antithetic(S0, K, r, sigma, T, N, M=252, option_type='call', avg_type='arithmetic'):
    """
    蒙特卡洛定价亚式期权（对偶变量法）
    """
    if N % 2 != 0:
        N = N + 1
    
    dt = T / M
    half_N = N // 2
    
    # 生成一半的随机数
    Z = np.random.standard_normal((half_N, M))
    
    # 正向路径
    paths_pos = np.zeros((half_N, M + 1))
    paths_pos[:, 0] = S0
    
    # 反向路径
    paths_neg = np.zeros((half_N, M + 1))
    paths_neg[:, 0] = S0
    
    for t in range(M):
        drift = (r - 0.5 * sigma**2) * dt
        vol = sigma * np.sqrt(dt)
        
        paths_pos[:, t + 1] = paths_pos[:, t] * np.exp(drift + vol * Z[:, t])
        paths_neg[:, t + 1] = paths_neg[:, t] * np.exp(drift - vol * Z[:, t])
    
    # 计算平均价格
    price_pos = paths_pos[:, 1:]
    price_neg = paths_neg[:, 1:]
    
    if avg_type == 'arithmetic':
        avg_pos = price_pos.mean(axis=1)
        avg_neg = price_neg.mean(axis=1)
    else:  # geometric
        avg_pos = np.exp(np.log(price_pos).mean(axis=1))
        avg_neg = np.exp(np.log(price_neg).mean(axis=1))
    
    # 计算 payoff
    if option_type == 'call':
        payoff_pos = np.maximum(avg_pos - K, 0)
        payoff_neg = np.maximum(avg_neg - K, 0)
    else:
        payoff_pos = np.maximum(K - avg_pos, 0)
        payoff_neg = np.maximum(K - avg_neg, 0)
    
    # 对偶变量法：取平均
    payoffs_avg = (payoff_pos + payoff_neg) / 2
    
    discount = np.exp(-r * T)
    price = discount * payoffs_avg.mean()
    std_error = discount * payoffs_avg.std() / np.sqrt(half_N)
    
    return {
        'price': price,
        'std_error': std_error,
        'ci_lower': price - 1.96 * std_error,
        'ci_upper': price + 1.96 * std_error
    }


# 对比标准 MC 和对偶变量法
np.random.seed(42)
arithmetic_antithetic = mc_asian_antithetic(S0, K, r, sigma, T, N_mc, avg_type='arithmetic')

print(f"方差缩减效果对比（算术平均亚式 Call，N = {N_mc:,}）：\n")
print(f"{'方法':<15} {'价格':>10} {'标准误差':>10} {'效率提升':>10}")
print("-" * 50)
print(f"{'标准 MC':<15} {arithmetic_asian['price']:>10.4f} {arithmetic_asian['std_error']:>10.4f} {'基准':>10}")

efficiency = (arithmetic_asian['std_error'] / arithmetic_antithetic['std_error'])**2
print(f"{'对偶变量法':<15} {arithmetic_antithetic['price']:>10.4f} {arithmetic_antithetic['std_error']:>10.4f} {efficiency:>9.2f}x")

print(f"\n解读：")
print(f"  - 对偶变量法在亚式期权定价中同样有效")
print(f"  - 效率提升约 {efficiency:.1f} 倍")

---
## 5. 观察频率的影响

### 5.1 不同观察频率

亚式期权的观察频率会影响价格：

| 观察频率 | 观察次数（1年） | 特点 |
|----------|----------------|------|
| 日均 | 252 次 | 最平滑，最接近连续观察 |
| 周均 | 52 次 | 中等平滑 |
| 月均 | 12 次 | 波动较大，价格较高 |
| 连续 | ∞ | 理论值，实际不可行 |

### 5.2 观察频率与期权价格的关系

In [ ]:
# 不同观察频率
frequencies = {
    '日均 (252)': 252,
    '周均 (52)': 52,
    '月均 (12)': 12,
    '季均 (4)': 4,
    '半年 (2)': 2
}

results = []
for name, M in frequencies.items():
    np.random.seed(42)
    result = mc_asian_option(S0, K, r, sigma, T, N_mc, M=M, avg_type='arithmetic')
    results.append({
        'name': name,
        'M': M,
        'price': result['price'],
        'std_error': result['std_error']
    })

# 绘制观察频率对价格的影响
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

M_values = [r['M'] for r in results]
prices = [r['price'] for r in results]
errors = [r['std_error'] for r in results]

# 左图：价格 vs 观察频率
axes[0].plot(M_values, prices, 'b-o', markersize=8, linewidth=2)
axes[0].axhline(y=geometric_asian, color='green', linestyle='--', 
                label=f'几何平均解析解 ({geometric_asian:.4f})')
axes[0].axhline(y=european_call, color='red', linestyle='--', 
                label=f'欧式 Call ({european_call:.4f})')

axes[0].set_xlabel('观察次数 M', fontsize=12)
axes[0].set_ylabel('期权价格', fontsize=12)
axes[0].set_title('算术平均亚式 Call 价格 vs 观察频率', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# 右图：标准误差 vs 观察频率
axes[1].plot(M_values, errors, 'r-s', markersize=8, linewidth=2)

axes[1].set_xlabel('观察次数 M', fontsize=12)
axes[1].set_ylabel('标准误差', fontsize=12)
axes[1].set_title('标准误差 vs 观察频率', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

print("\n不同观察频率的算术平均亚式 Call 价格：\n")
print(f"{'观察频率':<15} {'观察次数':>8} {'价格':>10} {'标准误差':>10}")
print("-" * 50)
for r in results:
    print(f"{r['name']:<15} {r['M']:>8} {r['price']:>10.4f} {r['std_error']:>10.4f}")

print(f"\n解读：")
print(f"  - 观察频率越高（M 越大），价格越接近几何平均亚式期权")
print(f"  - 观察频率越低（M 越小），价格越接近欧式期权")
print(f"  - 月均（M=12）已经能很好地近似连续观察")

---
## 6. 不同行权价的亚式期权

### 6.1 期权价格曲线

让我们比较欧式期权和亚式期权在不同行权价下的价格。

In [ ]:
# 不同行权价
K_values = np.linspace(80, 120, 9)

european_prices = []
arithmetic_asian_prices = []
geometric_asian_prices = []

for K_strike in K_values:
    # 欧式 Call
    european_prices.append(bs_call(S0, K_strike, r, sigma, T))
    
    # 几何平均亚式 Call（解析解）
    geometric_asian_prices.append(asian_geometric_call(S0, K_strike, r, sigma, T))
    
    # 算术平均亚式 Call（MC）
    np.random.seed(42)
    result = mc_asian_option(S0, K_strike, r, sigma, T, N_mc, avg_type='arithmetic')
    arithmetic_asian_prices.append(result['price'])

# 绘制价格曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：价格对比
axes[0].plot(K_values, european_prices, 'b-o', label='欧式 Call', markersize=6)
axes[0].plot(K_values, arithmetic_asian_prices, 'r-s', label='算术平均亚式', markersize=6)
axes[0].plot(K_values, geometric_asian_prices, 'g--^', label='几何平均亚式', markersize=6)
axes[0].axvline(x=S0, color='gray', linestyle=':', alpha=0.5, label=f'当前价格 S0={S0}')

axes[0].set_xlabel('行权价 K', fontsize=12)
axes[0].set_ylabel('期权价格', fontsize=12)
axes[0].set_title('不同行权价的期权价格', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 右图：亚式 vs 欧式的差异
diff_arithmetic = np.array(arithmetic_asian_prices) - np.array(european_prices)
diff_geometric = np.array(geometric_asian_prices) - np.array(european_prices)

axes[1].plot(K_values, diff_arithmetic, 'r-s', label='算术平均 - 欧式', markersize=6)
axes[1].plot(K_values, diff_geometric, 'g--^', label='几何平均 - 欧式', markersize=6)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axvline(x=S0, color='gray', linestyle=':', alpha=0.5)

axes[1].set_xlabel('行权价 K', fontsize=12)
axes[1].set_ylabel('价格差异', fontsize=12)
axes[1].set_title('亚式 vs 欧式的价格差异', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n解读：")
print("  - 亚式期权始终比欧式期权便宜")
print("  - 平值期权（K=S0）的差异最大")
print("  - 算术平均 > 几何平均（AM-GM 不等式）")
print("  - 深度实值/虚值期权的差异减小")

---
## 7. 实战：亚式期权的应用场景

### 7.1 商品价格风险管理

假设一家航空公司需要对冲未来一年的燃油成本：

- 每月购买一次燃油
- 总成本是 12 个月的平均价格
- 用亚式期权对冲平均价格风险

### 7.2 员工股票期权

很多公司的员工期权采用：

- 行权价 = 授予日后的平均价格
- 这实际上是一种亚式期权
- 平均化减少了市场波动的影响

In [ ]:
# 模拟对冲场景
def simulate_hedging_scenario(S0, r, sigma, T, M, N):
    """
    模拟航空公司对冲燃油成本的场景
    """
    # 模拟路径
    paths = simulate_gbm_paths(S0, r, sigma, T, M, N)
    
    # 计算每月平均价格（假设每月初购买）
    monthly_indices = np.linspace(0, M, 13, dtype=int)[:-1]  # 0, 21, 42, ..., 231
    monthly_prices = paths[:, monthly_indices]
    
    # 年平均价格
    annual_avg = monthly_prices.mean(axis=1)
    
    # 未对冲的成本（假设购买固定数量）
    unhedged_cost = annual_avg  # 平均购买价格
    
    # 用亚式 Call 对冲（行权价 = 当前价格）
    K_hedge = S0
    asian_call_payoff = np.maximum(annual_avg - K_hedge, 0)
    
    # 对冲后的成本
    hedged_cost = annual_avg - asian_call_payoff  # 期权收益抵消部分成本
    
    return {
        'unhedged_cost': unhedged_cost,
        'hedged_cost': hedged_cost,
        'asian_call_payoff': asian_call_payoff,
        'annual_avg': annual_avg
    }


# 模拟
np.random.seed(42)
N_sim = 50000
M_sim = 252

# 需要先定义 simulate_gbm_paths
def simulate_gbm_paths(S0, r, sigma, T, M, N):
    dt = T / M
    Z = np.random.standard_normal((N, M))
    paths = np.zeros((N, M + 1))
    paths[:, 0] = S0
    for t in range(M):
        paths[:, t + 1] = paths[:, t] * np.exp(
            (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t]
        )
    return paths

hedging_result = simulate_hedging_scenario(S0, r, sigma, T, M_sim, N_sim)

# 绘制结果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：成本分布
axes[0].hist(hedging_result['unhedged_cost'], bins=50, alpha=0.7, label='未对冲', color='red')
axes[0].hist(hedging_result['hedged_cost'], bins=50, alpha=0.7, label='对冲后', color='green')
axes[0].axvline(x=S0, color='blue', linestyle='--', linewidth=2, label=f'当前价格 {S0}')

axes[0].set_xlabel('平均购买成本', fontsize=12)
axes[0].set_ylabel('频次', fontsize=12)
axes[0].set_title('对冲前后成本分布', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 右图：对冲效果
cost_reduction = hedging_result['unhedged_cost'] - hedging_result['hedged_cost']
axes[1].hist(cost_reduction, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='无对冲收益')

axes[1].set_xlabel('成本减少（对冲收益）', fontsize=12)
axes[1].set_ylabel('频次', fontsize=12)
axes[1].set_title('亚式 Call 对冲收益分布', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n对冲效果分析：")
print(f"  未对冲平均成本: {hedging_result['unhedged_cost'].mean():.2f}")
print(f"  对冲后平均成本: {hedging_result['hedged_cost'].mean():.2f}")
print(f"  平均成本减少:   {cost_reduction.mean():.2f}")
print(f"  成本减少占比:   {cost_reduction.mean() / hedging_result['unhedged_cost'].mean() * 100:.1f}%")
print(f"\n解读：")
print(f"  - 亚式 Call 在价格上升时提供保护")
print(f"  - 对冲后成本分布更集中，波动更小")
print(f"  - 这就是为什么航空公司喜欢用亚式期权对冲")

---
## 8. MC 方法的优势：复杂衍生品

### 8.1 为什么 MC 适合复杂衍生品？

| 复杂性 | MC 的优势 |
|--------|----------|
| **路径依赖** | 自然处理，无需修改算法 |
| **多资产** | 可以模拟相关资产 |
| **奇异 payoff** | 只需修改 payoff 计算 |
| **高维** | 不受维度诅咒影响 |

### 8.2 MC vs 其他方法

| 方法 | 欧式期权 | 亚式期权 | 障碍期权 | 篮子期权 |
|------|----------|----------|----------|----------|
| **BS 公式** | ✓ 解析解 | ✗ 无解析解 | 部分 | ✗ |
| **二叉树** | ✓ | ✓（慢） | ✓ | ✓（维度诅咒） |
| **有限差分** | ✓ | ✓（复杂） | ✓ | ✗ |
| **蒙特卡洛** | ✓ | ✓ | ✓ | ✓ |

**结论**：MC 是复杂衍生品的「万能方法」，但计算成本较高。

---
## 9. 小结

### 核心收获

1. **亚式期权**是路径依赖期权，payoff 取决于平均价格

2. **算术平均 vs 几何平均**：
   - 算术平均更符合实际，但无解析解
   - 几何平均有解析解（波动率 = σ/√3）
   - 算术平均 ≥ 几何平均（AM-GM 不等式）

3. **观察频率**影响期权价格：
   - 频率越高，价格越低（更平滑）
   - 月均已能很好地近似连续观察

4. **MC 方法**是复杂衍生品的「万能方法」：
   - 自然处理路径依赖
   - 可以处理多资产、奇异 payoff
   - 可以与方差缩减技术结合

5. **亚式期权的应用**：
   - 商品价格风险管理
   - 员工股票期权
   - 对冲平均风险

### 关键公式

- 算术平均：$\bar{S} = \frac{1}{n} \sum S_{t_i}$
- 几何平均：$\bar{S} = (\prod S_{t_i})^{1/n}$
- 几何平均波动率：$\sigma_G = \sigma / \sqrt{3}$
- 几何平均亚式 Call：$C = e^{-rT} [F_0 N(d_1) - K N(d_2)]$

### 延伸阅读

- [Asian Option](https://en.wikipedia.org/wiki/Asian_option) - Wikipedia
- [Options, Futures, and Other Derivatives](https://www.amazon.com/Options-Futures-Other-Derivatives-10th/dp/013447208X) - John Hull

---
## 验收标准 Checklist

完成本次学习后，你应该能够：

- [x] **能处理路径依赖期权**：
  - 实现了 `mc_asian_option()` 函数
  - 理解了算术平均和几何平均的区别
  - 对比了不同观察频率的影响
- [x] **理解 MC 对复杂衍生品的优势**：
  - MC 自然处理路径依赖，无需修改算法
  - 可以处理多资产、奇异 payoff
  - 是复杂衍生品的「万能方法」

### 自测题

1. 为什么几何平均亚式期权有解析解，而算术平均没有？
2. 为什么亚式期权比欧式期权便宜？
3. 观察频率如何影响亚式期权的价格？为什么？
4. 列举两个亚式期权的实际应用场景。

---

**恭喜你完成了亚式期权定价的学习！** 🎉

下次我们将学习**VaR / CVaR**，了解如何度量投资组合的风险。